# 06.1 · RAG generator size: Qwen2.5 1.5B vs 3B

**Owner:** Member D

**Question.** In notebook 06 every demo synthesis fell back to copied source sentences, because Qwen2.5-1.5B ignored the instruction to cite a paper in every sentence (7 of 8 dropped sentences had no citation). Does a larger generator follow the citation instruction?

**Design.** Only the RAG generator changes. The retrieved papers, v2 prompt (`rag.build_rag_prompt(..., cite_every_sentence=True)`, notebook 05.1), system message (`rag.SYSTEM + rag.GUARD`), judge (Qwen2.5-1.5B, as in notebooks 05 and 06) and per-claim filter (`rag.filter_cited_claims`) are identical. Notebooks 05 and 06 are not changed or re-run: the 1.5B rows are answered from `artifacts/llm_cache.jsonl`, and the cell below checks that they are cached and that retrieval returns the same papers.

In [1]:
import pandas as pd

import llm
import rag
import retrieval
from utils import load_sample, load_split, load_results, retrieve, save_results

df = load_sample()
train_df, test_df = load_split(df)

SMALL = llm.MODEL                          # Qwen/Qwen2.5-1.5B-Instruct, the team default
LARGE = "Qwen/Qwen2.5-3B-Instruct"
SYSTEM = rag.SYSTEM + rag.GUARD            # as in notebooks 05 and 06
MAX_NEW_TOKENS = 450
print(llm.describe(), "| comparison model:", LARGE)

{'backend': 'hf', 'model': 'Qwen/Qwen2.5-1.5B-Instruct'} | comparison model: Qwen/Qwen2.5-3B-Instruct


## The same questions and papers as notebooks 05 and 06

Five research questions from the notebook 05 RAG ablation and the three notebook 06 demo queries. The pasted abstract uses `recommend(paper_id)`, so it cannot retrieve itself, exactly as in notebook 06.

In [2]:
RAG_QUESTIONS = ["How can large language models be defended against prompt injection attacks?",
                 "What methods improve sequential recommendation with transformers?",
                 "How is federated learning made robust to poisoned clients?",
                 "How do vision transformers compare with convolutional networks for image classification?",
                 "Which techniques reduce hallucination in retrieval-augmented question answering?"]
demo_paper = test_df.sample(n=1, random_state=7).iloc[0]
DEMO_QUERIES = [("defending large language models against prompt injection and jailbreak attacks", None),
                ("using graph neural networks to recommend items to users", None),
                (demo_paper["abstract"], demo_paper["id"])]

cases = [("NB05", q, retrieve(q, k=5)) for q in RAG_QUESTIONS]
cases += [("NB06", q, retrieval.recommend(pid, k=5) if pid else retrieve(q, k=5)) for q, pid in DEMO_QUERIES]

# Same retrieved papers as the saved notebook 05 and 06 runs
r05, r06 = load_results("05"), load_results("06")
assert [c[2]["id"].tolist() for c in cases[:5]] == [e["retrieved_ids"] for e in r05["rag_examples"]]
assert [c[2]["id"].tolist() for c in cases[5:]] == [d["paper_ids"] for d in r06["demo"]]
# The 1.5B answers are the cached ones from notebooks 05 and 06, not new generations
assert all(llm.is_cached(rag.build_rag_prompt(q, hits, cite_every_sentence=True), system=SYSTEM, max_new_tokens=MAX_NEW_TOKENS)
           for _, q, hits in cases)
print(len(cases), "questions; retrieval and 1.5B answers match notebooks 05 and 06")

8 questions; retrieval and 1.5B answers match notebooks 05 and 06


## Generate with each model, then apply the same checks

For every answer: format check, the judge's support for each synthesis sentence against all five excerpts (with the paper-count check), citation coverage, and the per-claim filter used by the demo. An answer whose filter keeps no sentence would fall back to copied sentences in the demo.

In [3]:
def evaluate(question, hits, model):
    reply = llm.generate(rag.build_rag_prompt(question, hits, cite_every_sentence=True), system=SYSTEM,
                         max_new_tokens=MAX_NEW_TOKENS, model=model)
    parts = rag.parse_sections(reply)
    supported, n = rag.groundedness(parts["synthesis"], rag.hits_source(hits), expected_papers=len(hits))
    citations = rag.validate_citations(parts["synthesis"], len(hits))
    check = rag.filter_cited_claims(parts["synthesis"], hits)   # judged by the default 1.5B model
    return {"reply": reply, "synthesis": parts["synthesis"], "format_ok": parts["format_ok"],
            "claims": n, "supported": supported, "cited": citations["cited_claims"],
            "invalid_citations": len(citations["invalid_references"]), "kept": len(check["kept"]),
            "kept_sentences": check["kept"], "dropped": check["dropped"], "fallback": not check["kept"]}


runs = []
for source, question, hits in cases:
    for model in (SMALL, LARGE):
        runs.append({"source": source, "question": question[:80], "model": model.split("/")[-1],
                     **evaluate(question, hits, model)})
runs = pd.DataFrame(runs)
print("done")

done


## Results

In [4]:
def summary(rows):
    claims = rows["claims"].sum()
    return pd.Series({"answers": len(rows),
                      "format ok": f"{rows['format_ok'].sum()}/{len(rows)}",
                      "synthesis sentences": claims,
                      "judge-supported": f"{rows['supported'].sum()}/{claims}",
                      "cited": f"{rows['cited'].sum()}/{claims}",
                      "kept by per-claim filter": f"{rows['kept'].sum()}/{claims}",
                      "invalid citations": rows["invalid_citations"].sum(),
                      "answers needing fallback": f"{rows['fallback'].sum()}/{len(rows)}"})


METRICS = ["claims", "format_ok", "supported", "cited", "kept", "invalid_citations", "fallback"]
table = runs.groupby(["source", "model"])[METRICS].apply(summary)
overall = runs.groupby("model")[METRICS].apply(summary)
display(table)
overall

answers format ok  synthesis sentences  \
source model                                                           
NB05   Qwen2.5-1.5B-Instruct        5       5/5                   21   
       Qwen2.5-3B-Instruct          5       5/5                   16   
NB06   Qwen2.5-1.5B-Instruct        3       3/3                    8   
       Qwen2.5-3B-Instruct          3       3/3                   13   

                             judge-supported  cited kept by per-claim filter  \
source model                                                                   
NB05   Qwen2.5-1.5B-Instruct           18/21  10/21                    10/21   
       Qwen2.5-3B-Instruct             13/16  13/16                    11/16   
NB06   Qwen2.5-1.5B-Instruct             5/8    1/8                      0/8   
       Qwen2.5-3B-Instruct             11/13  13/13                    12/13   

                              invalid citations answers needing fallback  
source model                                                              
NB05   Qwen2.5-1.5B-Instruct                  0                      2/5  
       Qwen2.5-3B-Instruct                    0                      0/5  
NB06   Qwen2.5-1.5B-Instruct                  0                      3/3  
       Qwen2.5-3B-Instruct                    0                      0/3

,answers,format ok,synthesis sentences,judge-supported,cited,kept by per-claim filter,invalid citations,answers needing fallback
model,,,,,,,,
Qwen2.5-1.5B-Instruct,8,8/8,29,23/29,11/29,10/29,0,5/8
Qwen2.5-3B-Instruct,8,8/8,29,24/29,26/29,23/29,0,0/8


In [5]:
# Per demo query: what each model's synthesis looks like after the per-claim filter
for question in runs.loc[runs["source"] == "NB06", "question"].unique():
    print("=" * 100)
    print("QUERY:", question)
    for _, r in runs[runs["question"] == question].iterrows():
        status = "FALLBACK" if r["fallback"] else f"{r['kept']} of {r['claims']} sentences kept"
        print(f"\n[{r['model']}] {status}")
        for sentence in r["kept_sentences"]:
            print("  kept:", sentence)
        for sentence, reason in r["dropped"]:
            print(f"  dropped ({reason}): {sentence[:110]}")

QUERY: defending large language models against prompt injection and jailbreak attacks

[Qwen2.5-1.5B-Instruct] FALLBACK
  dropped (no valid citation): All five papers collectively highlight the importance of studying both single-turn and multi-turn jailbreak at
  dropped (no valid citation): They emphasize the need for more comprehensive datasets and frameworks to address the weaknesses in current sa
  dropped (not supported by cited paper): Papers like [1], [2], and [3] specifically focus on introducing new techniques and datasets to improve LLM def

[Qwen2.5-3B-Instruct] 2 of 2 sentences kept
  kept: [1] [2] [3] [4] [5] The collective findings highlight the necessity of studying jailbreak attacks in both single and multi-turn formats, emphasizing the importance of considering input structure in defense strategies.
  kept: Additionally, [2] and [4] demonstrate the effectiveness of automated methods in generating and optimizing jailbreak prompts, while [5] provides a comprehensive eval

In [6]:
save_results("06_1", {
    "generators": [SMALL, LARGE], "judge": SMALL,
    "summary_by_source": table.reset_index().to_dict(orient="records"),
    "summary_overall": overall.reset_index().to_dict(orient="records"),
    "runs": runs.drop(columns=["reply"]).to_dict(orient="records"),
    "replies_large": runs.loc[runs["model"] == LARGE.split("/")[-1], ["question", "reply"]].to_dict(orient="records")})
print("Saved artifacts/06_1_results.json")

Saved artifacts/06_1_results.json


## Findings

- **Citations, not accuracy.** Over the same 8 questions, the 3B generator cited 26 of 29 synthesis sentences against 11 of 29 for 1.5B, and the per-claim filter kept 23 of 29 against 10 of 29. No 3B answer needed the extractive fallback (0/8, against 5/8 for 1.5B, including all three notebook 06 demos). The judge supported a similar share of sentences for both (24/29 vs 23/29), so the larger model is not more factual here; it follows the citation instruction, which makes its claims checkable.
- **What 3B writes.** For the two demo topics with clear papers it writes one cited sentence per paper, which is closer to an annotated reading list than a cross-paper synthesis. It also emitted a bare "[2]" and a generic sentence citing all five papers; the filter now drops content-free sentences, but a sentence citing every paper is checked against all five excerpts and is weak evidence.
- **Limits.** 8 questions; the judge is still the 1.5B model; gaps and recommendation reasons are not verified; 3B needs about twice the memory and generation time of 1.5B on the same laptop.
- **Decision.** Every evaluated number in notebooks 02–06 uses the 1.5B model, so it stays the team default. For the RAG step, 3B removes the demo's fallback and would be the generator of choice in a deployed system.